# Spacecraft Telemetry Anomaly Detection
## Stage 1 — Data Understanding & Preprocessing Pipeline

**Dataset v2:** 50 telemetry parameters · 161 commands · 10 000 rows  
**Tools:** NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn (scalers only)

---
# Section 1 — Project Introduction

## 1.1 What is Spacecraft Telemetry?
**Telemetry** is the automated process by which a spacecraft continuously measures its own internal parameters — battery voltage, temperatures, gyroscope readings, memory usage, RF signal strength — and transmits them to ground stations.

Each record captures **when** (`timestamp`), **what** was measured (`parameter`), and the **measured value** (`value`).

## 1.2 What are Telecommands?
**Telecommands** are instructions sent *from the ground station to the spacecraft*: activating payloads, orbit corrections, OBC reboots, attitude manoeuvres, etc. Each record: `timestamp`, `command`, `value` (1=success, 0=pending).

## 1.3 Why Anomaly Detection?
Spacecraft operate in extreme, unreachable environments. Anomaly detection enables:

| Challenge | How Detection Helps |
|-----------|--------------------|
| Hardware degradation | Early warning before failure |
| Sensor drift | Flag values deviating from norms |
| Software faults | Detect CPU / memory anomalies |
| Attitude failure | Identify unusual gyro / attitude readings |
| Power faults | Detect battery / solar panel anomalies |

## 1.4 Why Preprocessing?
- Raw telemetry is in **long format** — must be pivoted for ML
- Parameters span wildly different ranges — **scaling required**
- Timestamps are strings — must be parsed; **temporal features** extracted
- ML models need **feature-rich** representations: lag, rolling stats, change rates

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 140)

# Light theme
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.edgecolor':   '#dee2e6',
    'axes.labelcolor':  '#212529',
    'axes.titlecolor':  '#212529',
    'xtick.color':      '#495057',
    'ytick.color':      '#495057',
    'text.color':       '#212529',
    'grid.color':       '#dee2e6',
    'grid.linestyle':   '--',
    'grid.linewidth':   0.6,
    'font.family':      'DejaVu Sans',
    'font.size':        10,
    'axes.titlesize':   12,
    'axes.labelsize':   10,
})
sns.set_style('whitegrid')

PALETTE = ['#1f77b4','#2ca02c','#d62728','#9467bd',
           '#8c564b','#e377c2','#7f7f7f','#bcbd22',
           '#17becf','#ff7f0e']

os.makedirs('plots_v2', exist_ok=True)
os.makedirs('processed_v2', exist_ok=True)

print('Libraries loaded.')

---
# Section 2 — Dataset Loading

In [ ]:
telemetry_raw   = pd.read_csv('telemetry_train.csv')
telecommand_raw = pd.read_csv('telecommand_train.csv')

print('Telemetry  :', telemetry_raw.shape)
print('Telecommand:', telecommand_raw.shape)

In [ ]:
# Telemetry — head, info, describe
print('--- Telemetry head ---')
display(telemetry_raw.head(8))

print('\n--- Telemetry info ---')
telemetry_raw.info()

print('\n--- Telemetry describe ---')
display(telemetry_raw.describe())

print('\n--- Unique parameters ---')
print(sorted(telemetry_raw['parameter'].unique()))

In [ ]:
# Telecommand — head, info, describe
print('--- Telecommand head ---')
display(telecommand_raw.head(8))

print('\n--- Telecommand info ---')
telecommand_raw.info()

print('\n--- Telecommand describe ---')
display(telecommand_raw.describe())

print('\n--- Unique commands ---')
print(sorted(telecommand_raw['command'].unique()))

**Findings:**
- Telemetry: 10 000 rows, 3 columns, 50 unique parameters — all non-null, float values
- Telecommand: 161 rows, 3 columns, 161 unique commands — binary `value` (0/1), mean ≈ 0.96
- `timestamp` is a string type in both — must be parsed to datetime

---
# Section 3 — Data Quality Assessment

In [ ]:
def quality_report(df, name):
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    dups    = df.duplicated().sum()
    parsed  = pd.to_datetime(df['timestamp'], errors='coerce')
    bad_ts  = parsed.isnull().sum()
    print(f'[{name}]')
    print(f'  Rows            : {len(df):,}')
    print(f'  Duplicate rows  : {dups}')
    print(f'  Invalid timestamps: {bad_ts}')
    print(f'  Date range      : {parsed.min()} -> {parsed.max()}')
    print('  Missing values per column:')
    display(pd.DataFrame({'Count': missing, '%': pct})[missing > 0]
            if missing.any() else pd.DataFrame({'Result': ['None']}))
    print()

quality_report(telemetry_raw,   'TELEMETRY')
quality_report(telecommand_raw, 'TELECOMMAND')

In [ ]:
# Parameter / command name consistency
print('Telemetry whitespace anomalies  :',
      (telemetry_raw['parameter'].str.strip() != telemetry_raw['parameter']).sum())
print('Telecommand whitespace anomalies:',
      (telecommand_raw['command'].str.strip() != telecommand_raw['command']).sum())

# Per-parameter sample counts
param_counts = telemetry_raw['parameter'].value_counts()
print('\nSamples per parameter (min/max/mean):')
print(f'  Min  : {param_counts.min()}')
print(f'  Max  : {param_counts.max()}')
print(f'  Mean : {param_counts.mean():.1f}')

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Check':  ['Missing values','Duplicate rows','Invalid timestamps',
               'Whitespace anomalies','Parameter consistency'],
    'Telemetry':   ['None','None','None','None','Uniform round-robin'],
    'Telecommand': ['None','None','None','None','All prefixed CMD_'],
})
display(summary)

---
# Section 4 — Exploratory Data Analysis

In [ ]:
# 4.1 Parameter frequency
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

param_freq = telemetry_raw['parameter'].value_counts().sort_values()
param_freq.plot(kind='barh', ax=axes[0], color='#1f77b4', edgecolor='white')
axes[0].set_title('Telemetry Parameter Frequency', fontweight='bold')
axes[0].set_xlabel('Count')

cmd_freq = telecommand_raw['command'].value_counts().sort_values()
cmd_freq.plot(kind='barh', ax=axes[1], color='#2ca02c', edgecolor='white')
axes[1].set_title('Telecommand Frequency', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('plots_v2/01_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.2 Value distributions — all 50 parameters
params = sorted(telemetry_raw['parameter'].unique())
ncols  = 5
nrows  = (len(params) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(22, nrows * 3.5))
fig.suptitle('Value Distributions — All 50 Telemetry Parameters',
             fontsize=14, fontweight='bold', y=1.01)

for idx, param in enumerate(params):
    ax   = axes.flatten()[idx]
    data = telemetry_raw.loc[telemetry_raw['parameter'] == param, 'value']
    ax.hist(data, bins=25, color=PALETTE[idx % len(PALETTE)],
            alpha=0.80, edgecolor='white')
    ax.axvline(data.mean(),   color='red',    linewidth=1.2, linestyle='--')
    ax.axvline(data.median(), color='orange', linewidth=1.2, linestyle=':')
    ax.set_title(param, fontsize=7, fontweight='bold')
    ax.tick_params(labelsize=6)

for j in range(idx + 1, len(axes.flatten())):
    axes.flatten()[j].set_visible(False)

plt.tight_layout()
plt.savefig('plots_v2/02_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.3 Boxplots — per parameter (grouped by subsystem)
SUBSYSTEMS = {
    'Power':   [p for p in params if any(k in p for k in
                ['BATT','SOLAR','BUS'])],
    'Thermal': [p for p in params if 'TEMP' in p or 'RADIATOR' in p],
    'ADCS':    [p for p in params if any(k in p for k in
                ['GYRO','MAG','REACTION','ATTITUDE','STAR','SUN_SENSOR'])],
    'Comms':   [p for p in params if any(k in p for k in
                ['RF','TX','LINK','DATA_RATE','PACKET'])],
    'OBC':     [p for p in params if any(k in p for k in
                ['CPU','MEMORY','FLASH','WATCHDOG'])],
    'Propulsion': [p for p in params if any(k in p for k in
                   ['TANK','THRUSTER_VALVE','THRUSTER_TEMP'])],
}

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle('Boxplots by Subsystem', fontsize=13, fontweight='bold')

for ax, (subsys, ps), col in zip(axes.flatten(),
                                   SUBSYSTEMS.items(),
                                   PALETTE):
    data_list = [telemetry_raw.loc[telemetry_raw['parameter'] == p, 'value'].values
                 for p in ps if len(telemetry_raw[telemetry_raw['parameter'] == p]) > 0]
    labels    = [p for p in ps if len(telemetry_raw[telemetry_raw['parameter'] == p]) > 0]
    bp = ax.boxplot(data_list, labels=labels, patch_artist=True,
                    medianprops=dict(color='red', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor(col); patch.set_alpha(0.6)
    ax.set_title(subsys, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.tight_layout()
plt.savefig('plots_v2/03_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.4 Time-series — 8 representative parameters
tel_ts = telemetry_raw.copy()
tel_ts['timestamp'] = pd.to_datetime(tel_ts['timestamp'])

TS_PARAMS = ['BATT_VOLTAGE_1','SOLAR_POWER_TOTAL','OBC_TEMP','GYRO_X',
             'RF_SIGNAL_STRENGTH','ATTITUDE_ROLL','TANK_PRESSURE','MEMORY_USAGE']

fig, axes = plt.subplots(4, 2, figsize=(18, 14))
fig.suptitle('Time-Series Plots — Representative Parameters',
             fontsize=13, fontweight='bold')

for ax, param, col in zip(axes.flatten(), TS_PARAMS, PALETTE):
    sub  = tel_ts[tel_ts['parameter'] == param].sort_values('timestamp')
    ax.plot(sub['timestamp'], sub['value'],
            color=col, linewidth=0.8, alpha=0.75, label='Raw')
    rm   = sub['value'].rolling(window=10, center=True).mean()
    ax.plot(sub['timestamp'], rm,
            color='red', linewidth=1.5, linestyle='--', label='Rolling Mean (10)')
    ax.set_title(param, fontweight='bold', fontsize=9)
    ax.set_ylabel('Value', fontsize=8)
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('plots_v2/04_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

**EDA Observations:**
- All 50 parameters sampled at ~200 readings each (uniform round-robin polling)
- Most parameters follow approximately normal distributions — consistent with nominal operations
- RF signal, reaction wheel speeds, and struct temps show wider spread — reflecting orbital geometry variation
- Time-series reveal sinusoidal trends consistent with orbital day/night cycles
- Rolling mean tracks underlying trend; deviations from it are candidate anomaly signals

---
# Section 5 — Timestamp Processing

In [ ]:
telemetry   = telemetry_raw.copy()
telecommand = telecommand_raw.copy()

telemetry['timestamp']   = pd.to_datetime(telemetry['timestamp'])
telecommand['timestamp'] = pd.to_datetime(telecommand['timestamp'])

def add_temporal(df):
    ts = df['timestamp']
    df['hour']         = ts.dt.hour
    df['minute']       = ts.dt.minute
    df['second']       = ts.dt.second
    df['day']          = ts.dt.day
    df['month']        = ts.dt.month
    df['weekday']      = ts.dt.weekday
    df['is_weekend']   = (ts.dt.weekday >= 5).astype(int)
    df['minute_of_day']= ts.dt.hour * 60 + ts.dt.minute
    df['elapsed_sec']  = (ts - ts.min()).dt.total_seconds()
    return df

telemetry   = add_temporal(telemetry)
telecommand = add_temporal(telecommand)

print('Telemetry columns after timestamp processing:')
print(list(telemetry.columns))
display(telemetry.head(5))

In [ ]:
# Visualise temporal distributions
temporal_feats  = ['hour','minute','day','weekday','minute_of_day']
feature_labels  = ['Hour of Day','Minute','Day of Month',
                   'Day of Week (0=Mon)','Minute of Day']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Temporal Feature Distributions — Telemetry',
             fontsize=12, fontweight='bold')

for ax, feat, label, col in zip(axes, temporal_feats, feature_labels, PALETTE):
    ax.hist(telemetry[feat], bins=24, color=col, alpha=0.8, edgecolor='white')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel(label, fontsize=8)

plt.tight_layout()
plt.savefig('plots_v2/05_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

**Temporal features extracted:**

| Feature | Anomaly Relevance |
|---------|------------------|
| `hour` | Orbital day/night cycles — affects solar power & thermal |
| `minute`, `second` | Sub-hour telemetry burst patterns |
| `day`, `month` | Seasonal solar angle variation |
| `weekday`, `is_weekend` | Ground-station staffing patterns |
| `minute_of_day` | Full 1440-min daily cycle (LEO period ~90 min) |
| `elapsed_sec` | Absolute time position for trend detection |

---
# Section 6 — Feature Engineering

In [ ]:
tel_fe = telemetry.sort_values(['parameter', 'timestamp']).copy()
frames = []

for param, g in tel_fe.groupby('parameter'):
    g = g.copy()

    # Rolling statistics
    g['rolling_mean_5']      = g['value'].rolling(5,  min_periods=1).mean()
    g['rolling_mean_10']     = g['value'].rolling(10, min_periods=1).mean()
    g['rolling_std_5']       = g['value'].rolling(5,  min_periods=1).std().fillna(0)

    # Residual from rolling mean
    g['deviation_from_mean'] = g['value'] - g['rolling_mean_5']

    # Rate of change
    g['change_rate']         = g['value'].diff().fillna(0)
    g['abs_change_rate']     = g['change_rate'].abs()

    # Lag features
    g['lag_1'] = g['value'].shift(1).bfill()
    g['lag_2'] = g['value'].shift(2).bfill()
    g['lag_3'] = g['value'].shift(3).bfill()

    # Z-score (deviation in std units from local mean)
    std_safe   = g['rolling_std_5'].replace(0, np.nan)
    g['z_score'] = ((g['value'] - g['rolling_mean_5']) / std_safe).fillna(0)

    frames.append(g)

tel_fe = pd.concat(frames).sort_values('timestamp').reset_index(drop=True)

print('Feature-engineered shape:', tel_fe.shape)
print('Columns:', list(tel_fe.columns))
display(tel_fe[tel_fe['parameter'] == 'BATT_VOLTAGE_1'].head(8))

In [ ]:
# Feature summary table
feat_summary = pd.DataFrame({
    'Feature':     ['rolling_mean_5','rolling_mean_10','rolling_std_5',
                    'deviation_from_mean','change_rate','abs_change_rate',
                    'lag_1','lag_2','lag_3','z_score'],
    'Description': [
        'Mean of last 5 readings (per parameter)',
        'Mean of last 10 readings — longer trend baseline',
        'Std of last 5 readings — local volatility',
        'value - rolling_mean_5 — direct residual',
        'First difference — rate of change',
        'Absolute first difference — unsigned magnitude',
        'Value 1 step ago',
        'Value 2 steps ago',
        'Value 3 steps ago',
        'Deviation in standard deviation units from local mean',
    ],
    'Anomaly Signal': [
        'Large deviation from mean','Trend drift','Volatility spike',
        'Direct magnitude of anomaly','Sudden jump/drop','Unsigned severity',
        'Breaks autocorrelation','Breaks autocorrelation','Breaks autocorrelation',
        '|z|>2 or |z|>3 classical threshold',
    ]
})
display(feat_summary)

In [ ]:
# Visualise engineered features for one parameter
batt = tel_fe[tel_fe['parameter'] == 'BATT_VOLTAGE_1'].sort_values('timestamp')

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('Engineered Features — BATT_VOLTAGE_1', fontsize=13, fontweight='bold')

axes[0].plot(batt['timestamp'], batt['value'],
             color='#1f77b4', lw=0.9, label='Raw')
axes[0].plot(batt['timestamp'], batt['rolling_mean_5'],
             color='red', lw=1.5, ls='--', label='RM(5)')
axes[0].plot(batt['timestamp'], batt['rolling_mean_10'],
             color='green', lw=1.5, ls=':', label='RM(10)')
axes[0].legend(fontsize=8); axes[0].set_ylabel('Voltage (V)')
axes[0].set_title('Raw + Rolling Means', fontsize=10)

axes[1].fill_between(batt['timestamp'], batt['rolling_std_5'],
                     alpha=0.4, color='#9467bd')
axes[1].plot(batt['timestamp'], batt['rolling_std_5'],
             color='#9467bd', lw=1)
axes[1].set_ylabel('Std Dev'); axes[1].set_title('Rolling Std (5)', fontsize=10)

colors = ['#2ca02c' if v >= 0 else '#d62728' for v in batt['change_rate']]
axes[2].bar(batt['timestamp'], batt['change_rate'],
            color=colors, alpha=0.7, width=0.008)
axes[2].axhline(0, color='black', lw=0.5)
axes[2].set_ylabel('Delta'); axes[2].set_title('Change Rate (1st Difference)', fontsize=10)

axes[3].plot(batt['timestamp'], batt['z_score'],
             color='#1f77b4', lw=0.8)
for thr, ls, label in [(2,'--','|z|=2'),(3,':','|z|=3')]:
    axes[3].axhline( thr, color='red', ls=ls, lw=1.2, label=label)
    axes[3].axhline(-thr, color='red', ls=ls, lw=1.2)
axes[3].legend(fontsize=8)
axes[3].set_ylabel('Z-Score'); axes[3].set_xlabel('Timestamp')
axes[3].set_title('Z-Score', fontsize=10)
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

plt.tight_layout()
plt.savefig('plots_v2/06_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Section 7 — Feature Selection Analysis

In [ ]:
NUMERIC_COLS = ['value','hour','minute','day','weekday','elapsed_sec',
                'minute_of_day','rolling_mean_5','rolling_std_5',
                'rolling_mean_10','deviation_from_mean',
                'change_rate','abs_change_rate',
                'lag_1','lag_2','lag_3','z_score']

# Use one parameter for a meaningful within-parameter correlation
sample_param = tel_fe[tel_fe['parameter'] == 'OBC_TEMP'][NUMERIC_COLS].dropna()
corr = sample_param.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.4, annot_kws={'size': 7},
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — OBC_TEMP Features', fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('plots_v2/07_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# High-correlation pairs
THRESHOLD = 0.90
pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        v = corr.iloc[i, j]
        if abs(v) > THRESHOLD:
            pairs.append({'Feature A': corr.columns[i],
                          'Feature B': corr.columns[j],
                          'Correlation': round(v, 4)})

if pairs:
    hc = pd.DataFrame(pairs).sort_values('Correlation', key=abs, ascending=False)
    display(hc)
else:
    print(f'No pairs with |r| > {THRESHOLD}')

In [ ]:
# Variance analysis
var_df = sample_param.var().sort_values(ascending=False).reset_index()
var_df.columns = ['Feature', 'Variance']

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(var_df['Feature'], var_df['Variance'],
        color=[PALETTE[i % len(PALETTE)] for i in range(len(var_df))])
ax.set_xscale('log')
ax.set_title('Feature Variance (log scale) — OBC_TEMP', fontweight='bold')
ax.set_xlabel('Variance (log)')
ax.axvline(0.001, color='red', ls='--', lw=1.2, label='Low-variance threshold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots_v2/08_variance.png', dpi=150, bbox_inches='tight')
plt.show()

display(var_df.round(6))

**Feature Selection Findings:**

- `value`, `rolling_mean_5`, `rolling_mean_10`, `lag_1/2/3` — highly correlated (expected)
- **Recommended set for classical ML:** `value`, `rolling_mean_5`, `rolling_std_5`, `deviation_from_mean`, `change_rate`, `abs_change_rate`, `z_score`, `lag_1`, `hour`, `minute_of_day`
- **For deep models (GRU/TCN):** include all lag and rolling features — they provide rich reconstruction targets
- Features with near-zero variance can be dropped before model training

---
# Section 8 — Dataset Transformation

In [ ]:
# Long format (current)
print('LONG FORMAT — current structure:')
print(f'  Shape: {telemetry[["timestamp","parameter","value"]].shape}')
display(telemetry[['timestamp','parameter','value']].head(8))

In [ ]:
# Pivot to wide format
tel_wide = telemetry.pivot_table(
    index='timestamp',
    columns='parameter',
    values='value',
    aggfunc='mean'
).reset_index()

tel_wide.columns.name = None
tel_wide = tel_wide.sort_values('timestamp').reset_index(drop=True)

print('WIDE FORMAT — after pivot:')
print(f'  Shape: {tel_wide.shape}  '
      f'(1 row = 1 timestamp, 1 col = 1 parameter)')
display(tel_wide.iloc[:5, :8])   # first 8 columns for readability

In [ ]:
# Fill NaNs — forward fill then back fill
param_cols = [c for c in tel_wide.columns if c != 'timestamp']

print('NaN before fill:', tel_wide[param_cols].isnull().sum().sum())
tel_wide[param_cols] = tel_wide[param_cols].ffill().bfill()
print('NaN after fill :', tel_wide[param_cols].isnull().sum().sum())

print(f'\nFinal wide format shape: {tel_wide.shape}')

In [ ]:
# Format comparison
comparison = pd.DataFrame({
    'Property':        ['Rows','Columns','ML-ready','NaN handling','Storage'],
    'Long Format':     ['10 000','3 (timestamp, parameter, value)',
                        'No','Not needed','Compact'],
    'Wide Format':     ['200 (unique timestamps)','51 (timestamp + 50 params)',
                        'Yes','Forward-fill required','Larger'],
})
display(comparison)

---
# Section 9 — Data Scaling

In [ ]:
X_raw = tel_wide[param_cols].values

std_scaler = StandardScaler()
X_std      = std_scaler.fit_transform(X_raw)

mm_scaler  = MinMaxScaler()
X_mm       = mm_scaler.fit_transform(X_raw)

df_std = pd.DataFrame(X_std, columns=param_cols)
df_mm  = pd.DataFrame(X_mm,  columns=param_cols)

print('=== Raw Stats ===')
display(tel_wide[param_cols].describe().round(3))

print('\n=== StandardScaler Stats ===')
display(df_std.describe().round(3))

print('\n=== MinMaxScaler Stats ===')
display(df_mm.describe().round(3))

In [ ]:
# Visual comparison for 5 selected parameters
SEL = ['BATT_VOLTAGE_1','SOLAR_POWER_TOTAL','OBC_TEMP',
       'RF_SIGNAL_STRENGTH','GYRO_X']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Scaling Comparison — Selected Parameters',
             fontsize=12, fontweight='bold')

for ax, data, title in zip(axes,
                            [tel_wide[SEL], df_std[SEL], df_mm[SEL]],
                            ['Raw Values', 'StandardScaler (z-score)',
                             'MinMaxScaler [0,1]']):
    for col, c in zip(SEL, PALETTE):
        ax.hist(data[col], bins=25, alpha=0.55, label=col.replace('_',' '), color=c)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value'); ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('plots_v2/09_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scaler comparison table
scaler_table = pd.DataFrame({
    'Property':       ['Formula','Output range','Mean after scaling',
                       'Std after scaling','When to use'],
    'StandardScaler': ['z = (x - mean) / std','Unbounded, centred 0',
                       '~0','~1',
                       'Gaussian-like data; Isolation Forest, OCSVM'],
    'MinMaxScaler':   ['x\' = (x - min) / (max - min)','[0, 1]',
                       'Dataset-dependent','Dataset-dependent',
                       'Neural networks (sigmoid/tanh activations); GRU/TCN'],
})
display(scaler_table)

---
# Section 10 — Readiness for Machine Learning

In [ ]:
model_table = pd.DataFrame({
    'Model':                   ['Isolation Forest','One-Class SVM',
                                'GRU Autoencoder','TCN Autoencoder','NCDE'],
    'Type':                    ['Ensemble Tree','Kernel SVM',
                                'Recurrent NN','Conv NN','ODE NN'],
    'Temporal':                ['No','No','Yes','Yes','Yes (continuous)'],
    'Irregular Sampling':      ['No','No','No','No','Yes'],
    'Complexity':              ['Low','Low','Medium','Medium','High'],
    'Stage':                   ['2 — Baseline','2 — Baseline',
                                '3','3','4 — Advanced'],
})
display(model_table)

### Isolation Forest
Builds random decision trees. Anomalies require fewer splits to isolate (short path = anomalous). **Advantages:** fast, few hyperparameters, scikit-learn native. **Limitations:** no temporal awareness, misses gradual drift.

### One-Class SVM
Learns a kernel-based boundary enclosing normal data. **Advantages:** strong theory, effective for tight clusters. **Limitations:** O(n²) kernel, sensitive to `nu`/`gamma`, no temporal awareness.

### GRU Autoencoder
Encoder-decoder RNN compresses sequences to a latent vector and reconstructs them. High reconstruction error → anomaly. **Advantages:** native time-series, captures multi-parameter dependencies. **Limitations:** slow training, hyperparameter-heavy, black-box.

### TCN Autoencoder
Dilated causal convolutions for sequence modelling — parallelisable. **Advantages:** fast training, stable gradients, long receptive field. **Limitations:** fixed receptive field, more hyperparameters.

### NCDE (Neural Controlled Differential Equations)
Models hidden state as a continuous ODE driven by the telemetry path. **Advantages:** handles irregular sampling naturally, continuous-time. **Limitations:** expensive ODE solver at each step, complex implementation.

---
# Section 11 — Final Conclusion

In [ ]:
# Save all processed outputs
tel_fe.to_csv('processed_v2/telemetry_engineered.csv', index=False)
tel_wide.to_csv('processed_v2/telemetry_wide.csv', index=False)

df_std_out = pd.DataFrame(X_std, columns=param_cols)
df_std_out.insert(0, 'timestamp', tel_wide['timestamp'].values)
df_std_out.to_csv('processed_v2/telemetry_standard_scaled.csv', index=False)

df_mm_out = pd.DataFrame(X_mm, columns=param_cols)
df_mm_out.insert(0, 'timestamp', tel_wide['timestamp'].values)
df_mm_out.to_csv('processed_v2/telemetry_minmax_scaled.csv', index=False)

telecommand.to_csv('processed_v2/telecommand_processed.csv', index=False)

print('Saved files:')
for f in sorted(os.listdir('processed_v2')):
    print(f'  {f:<50} {os.path.getsize(os.path.join("processed_v2", f)):>10,} bytes')

In [ ]:
# Final summary
summary = pd.DataFrame({
    'Stage':       ['Dataset','Data Quality','Preprocessing',
                    'Feature Engineering','Feature Selection','ML Readiness'],
    'Details':     [
        '10 000 telemetry rows | 50 parameters | 161 commands',
        'No missing values, duplicates, or invalid timestamps',
        'Timestamps parsed | 9 temporal features | long->wide pivot | scaled (2 methods)',
        '10 features per parameter: rolling stats, lags, z-score, change rate',
        'High-corr pairs identified | variance ranked | recommended feature set defined',
        '5 candidate models described | baseline -> advanced roadmap defined',
    ],
    'Status': ['Done']*6
})
display(summary)
print('\n[STAGE 1 COMPLETE] Dataset ready for anomaly model development.')